In [ ]:
# ========================================
# PLATFORM SETUP — Choose ONE option
# ========================================

# --- OPTION A: KAGGLE (default) ---
import os, sys
KAGGLE_DATASET = '/kaggle/input/brain-tumor-project-files'  # <-- adjust if needed
sys.path.append(KAGGLE_DATASET)
IMAGES_DIR = os.path.join(KAGGLE_DATASET, 'data', 'segmentation', 'images')
MASKS_DIR = os.path.join(KAGGLE_DATASET, 'data', 'segmentation', 'masks')
MODEL_DIR = '/kaggle/working/models'
RESULTS_DIR = '/kaggle/working/results'
BASE_DIR = '/kaggle/working'

# --- OPTION B: GOOGLE COLAB (uncomment below, comment out Option A) ---
# from google.colab import drive
# drive.mount('/content/drive')
# import os, sys
# os.chdir('/content/drive/MyDrive/brain-tumor-project/notebooks')
# BASE_DIR = os.path.dirname(os.getcwd())
# sys.path.append(BASE_DIR)
# IMAGES_DIR = os.path.join(BASE_DIR, 'data', 'segmentation', 'images')
# MASKS_DIR = os.path.join(BASE_DIR, 'data', 'segmentation', 'masks')
# MODEL_DIR = os.path.join(BASE_DIR, 'models')
# RESULTS_DIR = os.path.join(BASE_DIR, 'results')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete!')


# 13 - Brain Tumor Segmentation with Attention U-Net + EfficientNet Encoder

**Architecture:** Attention U-Net with **pretrained EfficientNetB3** encoder (ImageNet weights)

**What's new?** Attention gates at each skip connection learn to suppress irrelevant
background features and focus on tumor regions. This is especially effective for medical
images where tumors are small relative to the full scan.

**Why pretrained?** Our dataset is only 3,064 images. Pretrained ImageNet weights give the
encoder a massive head start — it already knows edges, textures, and shapes.

**Reference:** Oktay et al., "Attention U-Net: Learning Where to Look for the Pancreas", 2018

**Dataset:** 3,064 grayscale MRI + binary masks | **Loss:** Dice + Focal


In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import warnings; warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

IMG_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 80
BACKBONE = 'EfficientNetB3'


## 1. Load Segmentation Data

**IMPORTANT:** Images are loaded in **0-255 range** (NOT normalized to 0-1).
`tf.keras.applications.EfficientNetB3` has an internal `Rescaling(1/255)` layer
and expects raw pixel values. Feeding 0-1 images would result in near-zero inputs.

In [ ]:
images = []
masks = []

image_files = sorted(os.listdir(IMAGES_DIR), key=lambda x: int(os.path.splitext(x)[0]))
mask_files = sorted(os.listdir(MASKS_DIR), key=lambda x: int(os.path.splitext(x)[0]))
print(f'Found {len(image_files)} images, {len(mask_files)} masks')

for img_f, mask_f in tqdm(zip(image_files, mask_files), total=len(image_files), desc='Loading'):
    assert os.path.splitext(img_f)[0] == os.path.splitext(mask_f)[0], f'Mismatch: {img_f} vs {mask_f}'
    
    # Load grayscale image → stack to 3 channels (pretrained encoder expects 3ch)
    # KEEP 0-255 RANGE — EfficientNetB3 normalizes internally!
    img = Image.open(os.path.join(IMAGES_DIR, img_f)).convert('L')
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
    img_arr = np.array(img, dtype=np.float32)  # 0-255 range
    img_arr = np.stack([img_arr]*3, axis=-1)
    images.append(img_arr)
    
    # Load binary mask (0 or 1)
    mask = Image.open(os.path.join(MASKS_DIR, mask_f)).convert('L')
    mask = mask.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
    mask_arr = (np.array(mask, dtype=np.float32) > 127).astype(np.float32)
    masks.append(mask_arr)

X = np.array(images, dtype=np.float32)
y = np.expand_dims(np.array(masks, dtype=np.float32), axis=-1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Image value range: [{X_train.min():.0f}, {X_train.max():.0f}]  (should be 0-255)')
print(f'Tumor pixel ratio: {y_train.mean()*100:.2f}%')


## 2. Visualize Samples

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(3):
    idx = np.random.randint(0, len(X_train))
    axes[i,0].imshow(X_train[idx,:,:,0], cmap='gray'); axes[i,0].set_title('MRI', fontweight='bold'); axes[i,0].axis('off')
    axes[i,1].imshow(y_train[idx].squeeze(), cmap='gray'); axes[i,1].set_title('Mask', fontweight='bold'); axes[i,1].axis('off')
    axes[i,2].imshow(X_train[idx,:,:,0], cmap='gray'); axes[i,2].imshow(y_train[idx].squeeze(), cmap='Reds', alpha=0.5)
    axes[i,2].set_title('Overlay', fontweight='bold'); axes[i,2].axis('off')
    area = y_train[idx].mean()*100
    axes[i,3].text(0.5,0.5,f'Tumor: {area:.1f}%',transform=axes[i,3].transAxes,fontsize=16,ha='center',va='center',fontweight='bold'); axes[i,3].axis('off')
plt.suptitle('Segmentation Data Samples', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'segmentation_samples.png'), dpi=150, bbox_inches='tight')
plt.show()


## 3. Data Augmentation

Augmentation adjusted for 0-255 pixel range.

In [ ]:
def augment(image, mask):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        mask = tf.image.flip_up_down(mask)
    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    mask = tf.image.rot90(mask, k)
    # Brightness/contrast adjusted for 0-255 range
    image = tf.image.random_brightness(image, max_delta=25.0)
    image = tf.image.random_contrast(image, 0.9, 1.1)
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, mask

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(2000)
train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
print(f'Train batches: {len(train_ds)} | Val batches: {len(val_ds)}')


## 4. Build Attention U-Net with Pretrained EfficientNetB3 Backbone

We use an EfficientNetB3 encoder pretrained on ImageNet and add **Attention Gates**
at each skip connection. The attention mechanism learns to:
- **Suppress** irrelevant background features (dark MRI background)
- **Amplify** tumor-relevant features (edges, textures at tumor boundaries)

**Key fix:** We use `input_tensor=` directly to avoid nested model wrapping issues
and ensure proper gradient flow through skip connections.

Formula: `α = σ(ψ(ReLU(W_x·x + W_g·g + b)))`, output = `x * α`

In [ ]:
# ── Attention Gate Module ──
def attention_gate(x, gating, inter_channels):
    """Additive attention gate for skip connections (Oktay et al., 2018)."""
    theta_x = layers.Conv2D(inter_channels, (1,1), padding='same')(x)
    theta_x = layers.BatchNormalization()(theta_x)
    phi_g = layers.Conv2D(inter_channels, (1,1), padding='same')(gating)
    phi_g = layers.BatchNormalization()(phi_g)
    add = layers.Add()([theta_x, phi_g])
    act = layers.Activation('relu')(add)
    psi = layers.Conv2D(1, (1,1), padding='same')(act)
    psi = layers.BatchNormalization()(psi)
    psi = layers.Activation('sigmoid')(psi)
    return layers.Multiply()([x, psi])

def conv_block(x, filters, dropout=0.0):
    """Double convolution block with BN."""
    x = layers.Conv2D(filters, (3,3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, (3,3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    if dropout > 0:
        x = layers.Dropout(dropout)(x)
    return x

# ── Build Attention U-Net with pretrained EfficientNetB3 encoder ──
def build_attention_unet_pretrained(input_shape=(256, 256, 3)):
    """
    Attention U-Net with pretrained EfficientNetB3 encoder.
    Uses input_tensor to avoid nested model issues.
    Images must be in 0-255 range (EfficientNet normalizes internally).
    """
    inputs = layers.Input(shape=input_shape, name='input_image')
    
    # Load pretrained encoder with shared input tensor
    encoder = tf.keras.applications.EfficientNetB3(
        weights='imagenet', include_top=False, input_tensor=inputs
    )
    
    # Extract skip connections at different spatial resolutions
    # For 256x256 input:
    s1 = encoder.get_layer('block2a_expand_activation').output   # 128x128
    s2 = encoder.get_layer('block3a_expand_activation').output   # 64x64
    s3 = encoder.get_layer('block4a_expand_activation').output   # 32x32
    s4 = encoder.get_layer('block6a_expand_activation').output   # 16x16
    bottleneck = encoder.output                                   # 8x8
    
    print(f'  Skip 1 (s1): {s1.shape}')
    print(f'  Skip 2 (s2): {s2.shape}')
    print(f'  Skip 3 (s3): {s3.shape}')
    print(f'  Skip 4 (s4): {s4.shape}')
    print(f'  Bottleneck:  {bottleneck.shape}')
    
    # ── Decoder with Attention Gates ──
    # Block 1: upsample bottleneck → 16x16, attend on s4
    u4 = layers.Conv2DTranspose(256, (2,2), strides=(2,2), padding='same')(bottleneck)
    a4 = attention_gate(s4, u4, inter_channels=128)
    d4 = layers.concatenate([u4, a4])
    d4 = conv_block(d4, 256, dropout=0.2)
    
    # Block 2: upsample d4 → 32x32, attend on s3
    u3 = layers.Conv2DTranspose(128, (2,2), strides=(2,2), padding='same')(d4)
    a3 = attention_gate(s3, u3, inter_channels=64)
    d3 = layers.concatenate([u3, a3])
    d3 = conv_block(d3, 128, dropout=0.2)
    
    # Block 3: upsample d3 → 64x64, attend on s2
    u2 = layers.Conv2DTranspose(64, (2,2), strides=(2,2), padding='same')(d3)
    a2 = attention_gate(s2, u2, inter_channels=32)
    d2 = layers.concatenate([u2, a2])
    d2 = conv_block(d2, 64, dropout=0.1)
    
    # Block 4: upsample d2 → 128x128, attend on s1
    u1 = layers.Conv2DTranspose(32, (2,2), strides=(2,2), padding='same')(d2)
    a1 = attention_gate(s1, u1, inter_channels=16)
    d1 = layers.concatenate([u1, a1])
    d1 = conv_block(d1, 32)
    
    # Final upsample → 256x256 (original resolution)
    u0 = layers.Conv2DTranspose(16, (2,2), strides=(2,2), padding='same')(d1)
    d0 = layers.Conv2D(16, (3,3), activation='relu', padding='same')(u0)
    d0 = layers.BatchNormalization()(d0)
    
    # Output
    outputs = layers.Conv2D(1, (1,1), activation='sigmoid', name='seg_output')(d0)
    
    model = models.Model(inputs, outputs, name='attention_unet_efficientnetb3')
    return model

model = build_attention_unet_pretrained(input_shape=(IMG_SIZE, IMG_SIZE, 3))

print(f'\nModel: Attention U-Net + {BACKBONE}')
print(f'Parameters: {model.count_params():,}')
total_params = model.count_params()
trainable_params = sum([K.count_params(w) for w in model.trainable_weights])
print(f'Trainable: {trainable_params:,}')


## 5. Loss: Dice + Focal

In [ ]:
# Custom Dice Loss
def dice_coefficient(y_true, y_pred, smooth=1.0):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2.*intersection+smooth) / (K.sum(y_true_f)+K.sum(y_pred_f)+smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coefficient(y_true, y_pred)

# Binary Focal Loss for class imbalance
def focal_loss(y_true, y_pred, alpha=0.75, gamma=2.0):
    y_pred = K.clip(y_pred, K.epsilon(), 1 - K.epsilon())
    pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
    alpha_t = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)
    return -K.mean(alpha_t * K.pow(1. - pt, gamma) * K.log(pt))

def total_loss(y_true, y_pred):
    return dice_loss(y_true, y_pred) + focal_loss(y_true, y_pred)

# IoU metric
def iou_score(y_true, y_pred, smooth=1.0):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(K.round(y_pred))
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

print('Loss: Dice Loss + Binary Focal Loss (alpha=0.75, gamma=2.0)')
print('Focal loss handles the extreme class imbalance (tumors are < 1% of pixels)')


## 6. Training

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=total_loss,
    metrics=[dice_coefficient, iou_score, 'accuracy']
)

callbacks = [
    ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'attention_unet_best.h5'),
        monitor='val_dice_coefficient', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(
        monitor='val_dice_coefficient', patience=20, restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7, min_lr=1e-7, verbose=1)
]

import time; start_time = time.time()
history = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds, callbacks=callbacks, verbose=1)
training_time = time.time() - start_time
print(f'\nDone in {training_time/60:.1f} min')


## 7. Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history.history['dice_coefficient'], label='Train', color='#FF6B6B', lw=2)
axes[0].plot(history.history['val_dice_coefficient'], label='Val', color='#4ECDC4', lw=2)
axes[0].set_title('Dice Coefficient', fontsize=14, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['iou_score'], label='Train', color='#FF6B6B', lw=2)
axes[1].plot(history.history['val_iou_score'], label='Val', color='#4ECDC4', lw=2)
axes[1].set_title('IoU Score', fontsize=14, fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[2].plot(history.history['loss'], label='Train', color='#FF6B6B', lw=2)
axes[2].plot(history.history['val_loss'], label='Val', color='#4ECDC4', lw=2)
axes[2].set_title('Loss', fontsize=14, fontweight='bold')
axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.suptitle('Attention U-Net + EfficientNetB3 Training', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'attention_unet_training_history.png'), dpi=150, bbox_inches='tight')
plt.show()


## 8. Evaluation

In [ ]:
best_path = os.path.join(MODEL_DIR, 'attention_unet_best.h5')
if os.path.exists(best_path):
    model = tf.keras.models.load_model(best_path, custom_objects={
        'total_loss': total_loss,
        'dice_coefficient': dice_coefficient,
        'iou_score': iou_score,
    })
    print('Loaded best model')

test_results = model.evaluate(X_test, y_test, verbose=1)
print(f'\n{"="*50}')
print('ATTENTION U-NET + EfficientNetB3 - TEST RESULTS')
print(f'{"="*50}')
print(f'Dice Coefficient: {test_results[1]:.4f}')
print(f'IoU Score:        {test_results[2]:.4f}')
print(f'Accuracy:         {test_results[3]:.4f}')

y_pred = model.predict(X_test, verbose=1)
y_pred_bin = (y_pred > 0.5).astype(np.float32)

# Per-sample Dice
dice_scores = []
for i in range(len(X_test)):
    gt=y_test[i].flatten(); pr=y_pred_bin[i].flatten()
    dice = (2.*np.sum(gt*pr)+1) / (np.sum(gt)+np.sum(pr)+1)
    dice_scores.append(dice)
print(f'\nPer-sample Dice: {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}')
print(f'Min: {np.min(dice_scores):.4f} | Max: {np.max(dice_scores):.4f}')


## 9. Prediction Visualization

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(16, 20))
indices = np.random.choice(len(X_test), 5, replace=False)
for row, idx in enumerate(indices):
    d = dice_scores[idx]
    axes[row,0].imshow(X_test[idx,:,:,0], cmap='gray'); axes[row,0].set_title('MRI', fontweight='bold'); axes[row,0].axis('off')
    axes[row,1].imshow(y_test[idx].squeeze(), cmap='gray'); axes[row,1].set_title('Ground Truth', fontweight='bold'); axes[row,1].axis('off')
    axes[row,2].imshow(y_pred_bin[idx].squeeze(), cmap='gray'); axes[row,2].set_title(f'Predicted (Dice:{d:.3f})', fontweight='bold'); axes[row,2].axis('off')
    axes[row,3].imshow(X_test[idx,:,:,0], cmap='gray'); axes[row,3].imshow(y_pred_bin[idx].squeeze(), cmap='Reds', alpha=0.4)
    axes[row,3].set_title('Overlay', fontweight='bold'); axes[row,3].axis('off')
plt.suptitle('Attention U-Net + EfficientNetB3 Predictions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'attention_unet_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()


## 10. Dice Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(dice_scores, bins=50, color='#4ECDC4', edgecolor='white', alpha=0.8)
ax.axvline(np.mean(dice_scores), color='red', ls='--', lw=2, label=f'Mean: {np.mean(dice_scores):.4f}')
ax.set_title('Per-Sample Dice Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Dice'); ax.set_ylabel('Count'); ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'attention_unet_dice_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()


## 11. Save Results

In [ ]:
import json as json_lib
seg_results = {
    'model': f'Attention U-Net + {BACKBONE}',
    'input_size': IMG_SIZE,
    'total_params': int(total_params),
    'trainable_params': int(trainable_params),
    'test_dice': round(float(test_results[1]), 4),
    'test_iou': round(float(test_results[2]), 4),
    'test_accuracy': round(float(test_results[3]), 4),
    'mean_sample_dice': round(float(np.mean(dice_scores)), 4),
    'epochs_trained': len(history.history['loss']),
    'training_time_sec': round(training_time, 1),
}
with open(os.path.join(RESULTS_DIR, 'attention_unet_results.json'), 'w') as f:
    json_lib.dump(seg_results, f, indent=2)
print('Results saved!')
print(f'Final Dice: {seg_results["test_dice"]}')
print(f'Final IoU:  {seg_results["test_iou"]}')
